# Build a tool-calling agent in Python (no SDK)

You will build a **weather assistant** with nothing but
[`requests`](https://requests.readthedocs.io/). No SDK. Every call is an HTTP request you can
read, copy into `curl`, and reason about.

The model decides when it needs the weather. Your Python runs the lookup, hands the result back,
and asks the model again. All of it lands in **one** trace.

| Piece | What it does | Who runs it |
|---|---|---|
| `get_weather` | the tool in the catalog: a name, a description, one `city` argument | the platform stores it |
| `WEATHER` and `get_weather_impl` | the Python that actually answers the lookup | **your code** |
| `py-weather-agent` | the prompt: the system message, a `city` template variable, the bound tool, the default model | the platform stores it |
| the loop in Step 8 | render, complete, run the tool, post its span, complete again | **your code** |

Every cell runs against a real account. Nothing here is faked or mocked.

**No SDK on purpose.** This notebook is the long way round, so you can see exactly what an SDK
does for you. If you want the short way, the same agent in six lines, read
[the SDK tutorial](https://docs.acruxcore.com/docs/tutorials/build-a-tool-calling-agent-in-python-sdk)
instead. Nothing here needs `pip install acruxcore`.

**Two ways to do every step.** Each step that creates something has two headings:
**In the dashboard**, with the values to type and a screenshot, and **The same thing in code**,
with a cell to run. They are not two different features — the dashboard and these `POST` calls
hit the same API, so the result is identical. Pick either. Doing both is harmless, because every
code cell looks for what already exists before it creates anything.

**Three kinds of code cell.** Most of this notebook is not the thing you would ship. Every cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |
| **Broken on purpose** | a failure being demonstrated | no |

**Companion page:** [Build a tool-calling agent in Python (no SDK)](https://docs.acruxcore.com/docs/tutorials/build-a-tool-calling-agent-in-python-no-sdk)

---

## Step 0 — What you need before you start

Three things on the platform, and one package.

**1. A credential.** A provider API key, encrypted at rest. **Gateway → Credentials → New
credential.** OpenRouter speaks the OpenAI protocol, so choose **OpenAI-compatible** — that
reveals a **Base URL** field.

| Field | What to enter |
|---|---|
| **Provider** | **OpenAI-compatible** |
| **Label** | `OpenRouter` |
| **API key** | your provider key — masked, and never shown again after you save |
| **Base URL** | `https://openrouter.ai/api/v1` |

![New credential dialog set to OpenAI-compatible, labelled OpenRouter, with a masked key and an OpenRouter base URL](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-no-sdk/01-new-credential.png)

**2. A model.** A credential on its own is not callable. A model is the public name your code
sends as `"model"`. **Gateway → Models → New model.**

| Field | What to enter |
|---|---|
| **Public name** | `llama-3.3-70b` — the name your code and your prompt will use |
| **Credential** | `OpenRouter`, the one you just created |
| **Upstream model** | `meta-llama/llama-3.3-70b-instruct`, the provider's own id |
| **Prices** | leave blank; they auto-fill for known models |

![New model dialog with public name llama-3.3-70b, an OpenRouter credential, and an upstream model id](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-no-sdk/02-new-model.png)

Click **Register model**, then **Test** on the new row. That fires a one-token completion and
tells you the key really works, before any of this notebook can blame itself.

**3. A personal API key.** **Account & keys → New key**, named `python-agent`. Copy it the
moment it appears — that is the only time the full value is shown.

**4. `requests`.** That is the only install.

In [ ]:
%pip install -q --upgrade requests

**Setup.** Set your key and the base URL, and name the things this notebook will create.

A key typed into a notebook is saved *inside the notebook file*. Prefer setting these in your
shell before you start Jupyter, and treat this cell as a fallback.

In [1]:
import json
import os

# Better: export these in your shell before starting Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")

API_KEY = os.environ["ACRUXCORE_API_KEY"]
BASE_URL = os.environ["ACRUXCORE_BASE_URL"].rstrip("/")

MODEL = "llama-3.3-70b"        # a public name from Gateway -> Models; any model works
TOOL = "get_weather"           # the tool this notebook creates
PROMPT = "py-weather-agent"    # the prompt this notebook creates

# Do NOT print BASE_URL: the saved output would publish whatever host you ran against.

### Preflight

**Check.** Run this before anything else. It checks things in the order they usually fail, so a
wrong key gives you one clear line instead of a `KeyError` from deep inside the loop.

One `requests.Session` is used for every call in this notebook. A session reuses the TCP
connection, which is worth doing in any real client, and it gives us one object to close at the
end.

In [2]:
import requests

http = requests.Session()
http.headers.update({
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
})


def api(method: str, path: str, body: dict | None = None) -> dict:
    """One JSON call against the API, raising on any non-2xx.

    A notebook helper, NOT an SDK. It only saves repeating the base URL and the
    raise_for_status() line - every call below is a plain HTTP request.
    """
    res = http.request(method, f"{BASE_URL}{path}", json=body, timeout=60)
    res.raise_for_status()
    return res.json() if res.content else {}


# 1. Does the key work at all?
api("GET", "/prompts?limit=1")
print("api key: ok")

# 2. Is there a model to run on, and is MODEL one of them?
models = [m["publicName"] for m in api("GET", "/gateway/models")]
print("models on this team:", models or "NONE - add one in Gateway -> Models")
print(f"MODEL {MODEL!r} available:", MODEL in models)

api key: ok
models on this team: ['mistral-small', 'llama-3.3-70b', 'claude-haiku', 'gemini-flash', 'gpt-4o-mini']
MODEL 'llama-3.3-70b' available: True


---

## Step 1 — The two ideas this page rests on

### Idea 1: a tool has two halves, and they live in different places

- The **definition** is what the model reads: a name, a description, a schema of arguments. It is
  text.
- The **implementation** is the code that does the work. The model never sees it.

AcruxCore always stores the definition in the **tool catalog**. Where the implementation lives is
the version's **executor**:

| Executor | Who runs the implementation | Good for |
|---|---|---|
| **HTTP** | the gateway. You describe a request once; the platform makes it. | a public or reachable API |
| **Client** | *your* process. The platform stores the schema and deliberately no body. | anything only your machine can reach |

This notebook uses **client**, because that is the realistic shape: the tool touches your
database, your internal service, or in our case a Python dict. So the catalog holds the `city`
schema, and your code holds the lookup.

### Idea 2: one trace across many separate HTTP calls

Here is the general problem. A tool-calling run is not one request. It is at least three: a model
turn, your tool, another model turn. Each of those is a separate HTTP call. Something has to say
"these belong together", or you get three unrelated records and no way to read the run.

AcruxCore does it with three headers on `POST /gateway/chat/completions`:

| Header | Direction | What it means |
|---|---|---|
| `x-trace-name` | you send it | open a **new** trace, and call it this |
| `x-gateway-trace-id` | the gateway sends it back | the id of the trace it used |
| `x-trace-id` | you send it | attach this call to **that existing** trace |

So the rule is: send `x-trace-name` on the first call only, keep the id that comes back, and send
it as `x-trace-id` on every call after that.

Your tool ran in your process, so the gateway knows nothing about it. You post that span
yourself, with `POST /traces`, carrying the same trace id.

### The trap

Forget to thread the id and nothing fails. There is no error and no warning. You simply get one
trace per model turn, each looking like a complete run that ends for no reason. Step 11 does this
on purpose so you can see what it looks like.

### The recommendation

Thread the header. If you would rather not hand-manage it, the Python SDK's
`run_prompt_with_tools` does this whole loop, including the tool span, in one call — that is
[the SDK tutorial](https://docs.acruxcore.com/docs/tutorials/build-a-tool-calling-agent-in-python-sdk).
Write it by hand once, here, so you know what it is doing.

---

## Step 2 — Create the `get_weather` shell

A tool in the catalog is **two objects**, not one. The **shell** owns the name and the
model-facing description. A **version** owns the argument schema and the executor. This step
creates the shell only.

A shell on its own is not callable. Step 3 makes it callable.

### In the dashboard

**Gateway → Tools → New tool.**

| Field | What to enter |
|---|---|
| **Name** | `get_weather` |
| **Description** | `Get the current weather for a city.` |

Those are the only two fields on the dialog. Click **Create tool**.

The description is the sentence the **model** reads when it decides whether to call this tool.
Write it for the model, not for your teammates.

### The same thing in code

**Setup.** `POST /tools`. Find-or-create: it looks the name up first, so a second run of this
notebook creates nothing.

In [3]:
TOOL_DESCRIPTION = "Get the current weather for a city."


def find_by_name(collection: str, name: str) -> dict | None:
    """The row in /tools or /prompts with exactly this name, or None.

    A notebook helper, NOT an SDK function. `?search=` matches substrings, so the
    exact-name filter has to happen here - `get_weather` would otherwise also match
    `get_weather_code`.
    """
    found = api("GET", f"/{collection}?search={name}&limit=100")
    return next((row for row in found["data"] if row["name"] == name), None)


tool = find_by_name("tools", TOOL)
if tool is None:
    tool = api("POST", "/tools", {"name": TOOL, "description": TOOL_DESCRIPTION})
    print(f"created tool shell {TOOL}")
else:
    print(f"tool {TOOL} already in the catalog")

print("tool id:", tool["id"])

created tool shell get_weather
tool id: 414c06d6-8411-448e-be02-407a64c76482


---

## Step 3 — Commit version 1, with a client executor

Now the shell gets its argument schema and its executor. `{"type": "client"}` is the sentence
"my own app runs this."

### In the dashboard

**Gateway → Tools → `get_weather` → New version.**

| Field | What to enter |
|---|---|
| **Parameters** | one row: name `city`, type `string`, **required** |
| **`city` description** | `City name, e.g. Tokyo.` |
| **Executor** | **Client — the caller's app runs it** |

Click **Commit version**. The first version automatically gets the `production` and `staging`
aliases. Later versions move no alias for you — you promote them yourself.

### The same thing in code

**Setup.** `POST /tools/<id>/versions`. A version is immutable, which is why changing a schema
means committing a new version rather than editing this one. This cell commits only when the
tool has no versions yet.

In [4]:
CITY_SCHEMA = {
    "type": "object",
    "properties": {
        "city": {"type": "string", "description": 'City name, e.g. "Tokyo".'}
    },
    "required": ["city"],
}

versions = api("GET", f"/tools/{tool['id']}/versions?limit=1")
if versions["total"] == 0:
    version = api("POST", f"/tools/{tool['id']}/versions", {
        "description": TOOL_DESCRIPTION,
        "parametersSchema": CITY_SCHEMA,
        "executor": {"type": "client"},
    })
    aliases = [a["alias"] for a in version["aliases"]]
    print(f"committed v{version['versionNumber']}, aliases now pointing here: {aliases}")
else:
    print(f"tool already has {versions['total']} version(s) - nothing committed")

committed v1, aliases now pointing here: ['production', 'staging']


**Check.** What the model will actually read, and under which executor. `POST /tools/resolve`
answers both in one call.

In [5]:
resolved = api("POST", "/tools/resolve", {"refs": [{"name": TOOL, "alias": "production"}]})
entry = resolved["data"][0]      # results come back in the order the refs were sent

print("executor:", entry["executorType"], "(client = your process runs it)")
print("version: ", entry["versionNumber"])
print(json.dumps(entry["function"], indent=2))

executor: client (client = your process runs it)
version:  1
{
  "name": "get_weather",
  "description": "Get the current weather for a city.",
  "parameters": {
    "type": "object",
    "required": [
      "city"
    ],
    "properties": {
      "city": {
        "type": "string",
        "description": "City name, e.g. \"Tokyo\"."
      }
    }
  }
}


---

## Step 4 — Create the prompt

The prompt holds the **system instructions** and the **default model**. It also holds a
`{{ city }}` **template variable** in the user message, which is the part worth pausing on.

There are two ways to get the user's question into a run, and this page uses the other one from
the SDK tutorial:

- **A template variable**, as here. The message shape is stored, and your code fills the blank at
  render time. Good when the question always has the same shape.
- **Appending a message in code.** Nothing is stored about the question. Good when it is free
  text.

### In the dashboard

**Prompts → New prompt**, then the **Editor** tab.

| Field | What to enter |
|---|---|
| **Name** | `py-weather-agent` |
| **Description** | `Weather assistant driven by a plain-Python REST tool loop.` |
| **Default model** | `llama-3.3-70b` |
| **System message** | the `SYSTEM` string in the next code cell |
| **User message** | `What is the weather in {{ city }} right now?` |

![The py-weather-agent prompt Editor tab showing default model llama-3.3-70b, a system and user message, and production pointing at v1](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-no-sdk/04-prompt-editor.png)

Click **Commit version**. The model is part of the version, so committing bakes it in, and
because it is the first commit `production` points at `v1` automatically.

### The same thing in code

**Setup.** Two separate checks on purpose. A prompt shell with zero versions is a real state — an
earlier run that died between the two calls leaves one — and "the name exists" is not "it has
content".

In [6]:
SYSTEM = (
    "You are a weather assistant. Use the get_weather tool to look up conditions "
    "before answering. Never guess."
)
USER = "What is the weather in {{ city }} right now?"

prompt = find_by_name("prompts", PROMPT)
if prompt is None:
    prompt = api("POST", "/prompts", {
        "name": PROMPT,
        "description": "Weather assistant driven by a plain-Python REST tool loop.",
    })
    print(f"created prompt shell {PROMPT}")
else:
    print(f"prompt {PROMPT} already exists")

if api("GET", f"/prompts/{prompt['id']}/versions?limit=1")["total"] == 0:
    pv = api("POST", f"/prompts/{prompt['id']}/versions", {
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": USER},
        ],
        "model": MODEL,          # baked into the version - your code never hardcodes it
    })
    print(f"committed prompt v{pv['versionNumber']} on model {pv['model']}")
else:
    print("prompt already has a version - nothing committed")

created prompt shell py-weather-agent
committed prompt v1 on model llama-3.3-70b


---

## Step 5 — Connect the tool to the prompt

Right now the tool and the prompt know nothing about each other. A **binding** joins them, so one
render call returns the messages *and* the tool schema together.

The binding stores an **alias**, not a version number. Point it at `production` and the prompt
follows whatever you promote to `production` later, with no code change.

### In the dashboard

**Prompts → `py-weather-agent` → Tools tab → + Connect a tool from the catalog.**

| Field | What to enter |
|---|---|
| **Tool** | `get_weather` |
| **Alias** | `production` |
| **Column** | **default** — every alias of the prompt inherits it |

![The prompt Tools tab showing get weather connected](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-no-sdk/05-prompt-tools.png)

It saves straight away; there is no separate commit for a binding.

### The same thing in code

**Setup.** `PUT`, not `POST`, and that matters: a `PUT` replaces the binding for that tool rather
than adding a second one, so running this twice leaves one row.

In [7]:
binding = api("PUT", f"/prompts/{prompt['id']}/tools/{tool['id']}", {"tool_alias": "production"})
print(f"bound {binding['toolName']} @ {binding['toolAlias']} -> "
      f"v{binding['resolvedVersionNumber']}")

bound get_weather @ production -> v1


**Check.** One render call, and everything the model needs comes back together, already in the
shape the OpenAI protocol wants. `variables` is where the `city` blank gets filled.

In [8]:
rendered = api("POST", f"/prompts/{PROMPT}/production/render", {"variables": {"city": "Tokyo"}})

print("model:   ", rendered["model"])
print("tools:   ", [t["function"]["name"] for t in rendered["tools"]])
for message in rendered["messages"]:
    print(f"  {message['role']}: {message['content'][:80]}")

model:    llama-3.3-70b
tools:    ['get_weather']
  system: You are a weather assistant. Use the get_weather tool to look up conditions befo
  user: What is the weather in Tokyo right now?


---

## Step 6 — Write the implementation

The catalog holds the schema and no body. This cell is the body.

**Your app.** The canned dict keeps the notebook self-contained. A real tool would call a weather
API, read a database, or hit an internal service — anything your process can do and the gateway
cannot.

Note the dispatch function. The model sends back a tool **name** as a string, so something has to
turn that string into a call. A dict of name to function is enough, and it fails loudly on a name
you did not expect.

In [9]:
WEATHER = {
    "tokyo": "22°C, light rain",
    "london": "15°C, overcast",
    "paris": "19°C, clear",
}


def get_weather_impl(city: str) -> dict:
    """The local implementation the model's `get_weather` tool call routes to."""
    return {"city": city, "conditions": WEATHER.get(city.lower(), "no data for that city")}


#: Keyed by catalog tool name. The catalog holds the schema and deliberately no body,
#: so this map is the whole of what your app contributes.
IMPLEMENTATIONS = {TOOL: get_weather_impl}


def run_tool(name: str, args: dict):
    """Route one tool call from the model to its local implementation."""
    if name not in IMPLEMENTATIONS:
        raise ValueError(f"the model asked for an unknown tool: {name}")
    return IMPLEMENTATIONS[name](**args)


print("implementations ready for:", list(IMPLEMENTATIONS))

implementations ready for: ['get_weather']


---

## Step 7 — The two calls the loop is built from

**Your app.** These two functions are where the trace threading from Step 1 actually happens.
Read `complete` first: the whole one-trace trick is the three lines that set a header.

`log_tool_span` is the other half. The gateway records the LLM spans on its own, because it made
those calls. Your tool ran in your process, so nobody but you can report it.

In [10]:
from datetime import datetime, timezone


def now() -> str:
    """Current time as an ISO-8601 string with a timezone offset, which is what /traces wants."""
    return datetime.now(timezone.utc).isoformat()


def complete(model: str, messages: list, tools: list, trace_id: str | None):
    """One gateway completion. Threads the trace, and returns (message, trace_id)."""
    headers = {}
    if trace_id:
        headers["x-trace-id"] = trace_id              # attach to the existing trace
    else:
        headers["x-trace-name"] = PROMPT              # open a new trace, named this

    res = http.post(
        f"{BASE_URL}/gateway/chat/completions",
        headers=headers,
        json={"model": model, "messages": messages, "tools": tools},
        timeout=120,
    )
    res.raise_for_status()
    # The gateway records the LLM span itself; we only need the trace id it used.
    return res.json()["choices"][0]["message"], res.headers["x-gateway-trace-id"]


def log_tool_span(trace_id: str, name: str, args: dict, result, started: str, ended: str) -> None:
    """Add a `tool` span for a client-side call to the SAME trace as the LLM spans."""
    api("POST", "/traces", {
        "traces": [{
            "traceId": trace_id,
            "capturePayloads": True,      # without this the span has no input/output
            "spans": [{
                "spanId": f"{name}-{started}",
                "name": name,
                "kind": "tool",
                "status": "ok",
                "startTime": started,
                "endTime": ended,
                "input": args,
                "output": result,
            }],
        }],
    })


print("loop helpers ready")

loop helpers ready


---

## Step 8 — Run the agent

**Your app.** This is the loop, and it is the whole point of the page. Four moves:

1. Render the stored prompt, filling the `city` blank.
2. Complete. On the first turn there is no trace id yet, so `complete` opens one.
3. If the model asked for tools, run each one, post its span to the same trace, and append the
   result as a `tool` message. The `tool_call_id` is not optional — it is how the model knows
   which of its requests this answers.
4. Loop. When a turn comes back with no tool calls, that is the answer.

The turn cap is there so a misbehaving model cannot loop forever. Keep one in your own code.

In [11]:
def ask_about(city: str, max_turns: int = 5) -> tuple[str, str]:
    """Answer one weather question with the stored prompt and the local tool."""
    rendered = api("POST", f"/prompts/{PROMPT}/production/render", {"variables": {"city": city}})
    messages, tools = rendered["messages"], rendered["tools"]
    print(f"rendered {len(messages)} message(s) + {len(tools)} tool(s) "
          f"[{', '.join(t['function']['name'] for t in tools)}]")

    trace_id = None
    for turn in range(1, max_turns + 1):
        message, trace_id = complete(rendered["model"], messages, tools, trace_id)
        messages.append(message)

        tool_calls = message.get("tool_calls")
        if not tool_calls:
            print(f"({turn} model turn(s), trace {trace_id})")
            return message["content"], trace_id

        for call in tool_calls:
            name = call["function"]["name"]
            args = json.loads(call["function"]["arguments"])
            started = now()
            result = run_tool(name, args)
            ended = now()
            print(f"  -> {name}({args}) = {result}")
            log_tool_span(trace_id, name, args, result, started, ended)
            messages.append({
                "role": "tool",
                "tool_call_id": call["id"],     # NOT optional: it pairs the answer to the request
                "content": json.dumps(result),
            })

    raise RuntimeError("hit the turn limit without a final answer")


answer, TRACE_ID = ask_about("Tokyo")
print("\nAssistant:", answer)

rendered 2 message(s) + 1 tool(s) [get_weather]
  -> get_weather({'city': 'Tokyo'}) = {'city': 'Tokyo', 'conditions': '22°C, light rain'}
(2 model turn(s), trace 485b1e28-98ea-4363-99f9-90a5a10641f2)

Assistant: The current weather in Tokyo is 22°C with light rain.


Two turns, one trace. The first turn produced no prose at all — only a request for
`get_weather` with `{'city': 'Tokyo'}`, which the model chose on its own from the schema. The
second turn is the sentence.

The conditions in the answer came out of your `WEATHER` dict. Nothing in the prompt or the
catalog knows them.

---

## Step 9 — Read the trace back

**Check.** Read it from the API rather than trusting a screenshot: `GET /traces/<id>` returns the
trace header plus every span assembled into a parent/child tree.

The **shape** is what to check — an LLM span that asked for the tool, your `get_weather` tool span,
and a final LLM span that wrote the answer. The gateway recorded the first and third; you posted
the second. Token counts, costs and timings move on every run, because the model does not answer
identically twice. Only the shape is stable.

In [12]:
detail = api("GET", f"/traces/{TRACE_ID}")


def walk(spans, depth=0):
    for span in spans:
        # An llm span is named by its timestamp, so the model is the useful label.
        label = span["model"] if span["kind"] == "llm" else span["name"]
        print(f"{'  ' * depth}- [{span['kind']}] {label}  {span['status']}  "
              f"tokens={span.get('totalTokens') or 0}")
        walk(span.get("children") or [], depth + 1)


print(f"trace: {detail['trace']['name']}")
print(f"spans: {detail['trace']['spanCount']}   total tokens: {detail['trace']['totalTokens']}")
walk(detail["spans"])

tool_span = next(s for s in detail["spans"] if s["kind"] == "tool")
print("\ntool span payload:", json.dumps(tool_span.get("payload"), indent=2))

trace: py-weather-agent
spans: 3   total tokens: 559
- [llm] meta-llama/llama-3.3-70b-instruct  ok  tokens=273
- [tool] get_weather  ok  tokens=0
- [llm] meta-llama/llama-3.3-70b-instruct  ok  tokens=286

tool span payload: {
  "input": {
    "city": "Tokyo"
  },
  "output": {
    "city": "Tokyo",
    "conditions": "22\u00b0C, light rain"
  },
  "variables": null
}


All three spans came back at the same level, as siblings. That is because the span you posted
carried no `parentSpanId`, so nothing said it happened *inside* the first model turn. The SDK
nests it for you; over REST you would set `parentSpanId` yourself. Sibling spans are perfectly
readable, so this is a choice rather than a bug — but know that you made it.

The payload is the other thing to look at. It is there because `log_tool_span` sent
`capturePayloads: True`. Without that the span still exists and still shows in the tree, but with
no input and no output — you would see *that* the tool ran and never *what it did*.

The dashboard shows the same tree, under **Observability → Traces**:

![Trace named py-weather-agent with three spans, an LLM span, a get weather tool span, and a final LLM span, all marked OK](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-no-sdk/06-trace-tree.png)

![The expanded get weather span showing an Input of city Tokyo and an Output with the conditions](https://docs.acruxcore.com/img/tutorials/build-a-tool-calling-agent-in-python-no-sdk/07-trace-span-payload.png)

Payloads are stored only while payload capture is on for your team as well. It is on by default;
turn it off in **Observability → Settings** if you would rather not store request bodies.

---

## Step 10 — Stream the reply

**Your app.** Sometimes you want tokens as they are generated. Set `"stream": true` in the body,
pass `stream=True` to `requests` so the socket stays open, and read the
[Server-Sent Events](https://developer.mozilla.org/docs/Web/API/Server-sent_events) frames: one
`data:` line per chunk, each carrying a `delta.content` string, ending with `data: [DONE]`.

Notice what this call does **not** send: `tools`. Streaming yields text deltas and does not run
tools for you. Forward the tools here and the first turn streams tool-call *fragments* instead of
prose. Step 11 shows that.

In [13]:
rendered = api("POST", f"/prompts/{PROMPT}/production/render", {"variables": {"city": "Paris"}})

res = http.post(
    f"{BASE_URL}/gateway/chat/completions",
    json={"model": rendered["model"], "messages": rendered["messages"], "stream": True},
    stream=True,          # keep the socket open so frames arrive as they are produced
    timeout=120,
)
res.raise_for_status()

full = ""
for line in res.iter_lines(decode_unicode=True):
    if not line or not line.startswith("data: "):
        continue
    data = line[len("data: "):]
    if data == "[DONE]":
        break
    piece = json.loads(data)["choices"][0]["delta"].get("content", "") or ""
    print(piece, end="", flush=True)
    full += piece

print(f"\n\n(streamed {len(full)} characters)")

Let me check the current weather in Paris for you.

(streamed 50 characters)


Read that answer against Step 8's. With no tool to call, the model can only say it *would* check
the weather. That is exactly why the loop exists: the loop returns conditions, streaming returns
prose.

---

## Step 11 — Four ways to get this wrong

Every cell in this step is **broken on purpose**. None of it is app code.

### Mistake 1 — you forget to thread the trace id

**Broken on purpose.** This is the quiet one, and it is the reason Step 1 spends so long on three
headers. Nothing errors. You just get a separate trace per model turn, each one looking like a
finished run that stops for no reason.

In [14]:
rendered = api("POST", f"/prompts/{PROMPT}/production/render", {"variables": {"city": "London"}})
messages, tools = rendered["messages"], rendered["tools"]

first, trace_a = complete(rendered["model"], messages, tools, None)
messages.append(first)
for call in first.get("tool_calls") or []:
    args = json.loads(call["function"]["arguments"])
    messages.append({
        "role": "tool",
        "tool_call_id": call["id"],
        "content": json.dumps(run_tool(call["function"]["name"], args)),
    })

# Broken on purpose: passing None again opens a SECOND trace instead of continuing the first.
second, trace_b = complete(rendered["model"], messages, tools, None)

print("trace after turn 1:", trace_a)
print("trace after turn 2:", trace_b)
print("same trace?        ", trace_a == trace_b)
print(f"\nspans in trace 1: {api('GET', f'/traces/{trace_a}')['trace']['spanCount']}")
print(f"spans in trace 2: {api('GET', f'/traces/{trace_b}')['trace']['spanCount']}")

trace after turn 1: eb12cb28-bb15-454c-9fa7-6b15b47ee229
trace after turn 2: 441141fa-ae14-45f6-9cdf-73561a7860d2
same trace?         False

spans in trace 1: 1
spans in trace 2: 1


Two traces of one span each, instead of one trace of three. No error was raised, nothing is red
in the dashboard, and the run is simply unreadable afterwards. Worse, neither half is wrong on its
own, so there is nothing to notice until someone asks why a tool call has no answer next to it.

### Mistake 2 — the tool result has no `tool_call_id`

**Broken on purpose.** The model can ask for several tools in one turn, so every answer has to
say which request it belongs to. Drop the id and the provider rejects the whole message list.

In [15]:
rendered = api("POST", f"/prompts/{PROMPT}/production/render", {"variables": {"city": "Tokyo"}})
messages, tools = rendered["messages"], rendered["tools"]
first, trace_id = complete(rendered["model"], messages, tools, None)
messages.append(first)

for call in first.get("tool_calls") or []:
    args = json.loads(call["function"]["arguments"])
    messages.append({
        "role": "tool",
        # Broken on purpose: no "tool_call_id" key at all.
        "content": json.dumps(run_tool(call["function"]["name"], args)),
    })

try:
    complete(rendered["model"], messages, tools, trace_id)
    print("no error - unexpected")
except requests.HTTPError as err:
    print(f"HTTP {err.response.status_code}")
    print(json.dumps(err.response.json(), indent=2)[:600])

HTTP 400
{
  "error": {
    "code": "VALIDATION_ERROR",
    "message": "A tool message requires tool_call_id."
  }
}


### Mistake 3 — the tool span goes to its own trace

**Broken on purpose.** `POST /traces` will happily create a trace it has never seen. Pass the
wrong id, or forget to pass one, and the span is stored perfectly — just nowhere near the model
turns it belongs to.

Watch the name it prints. A trace created this way was never given one, so it gets a timestamp,
which is how an accidental trace looks in the list.

In [16]:
import uuid

orphan_id = str(uuid.uuid4())     # broken on purpose: an id no completion ever used
started = now()
log_tool_span(orphan_id, TOOL, {"city": "Tokyo"}, get_weather_impl("Tokyo"), started, now())

orphan = api("GET", f"/traces/{orphan_id}")
print("the orphan trace exists:", orphan["trace"]["spanCount"], "span")
print("its name:", orphan["trace"]["name"])
print("\nand the real run still has:", api("GET", f"/traces/{TRACE_ID}")["trace"]["spanCount"],
      "spans - the tool call above is not one of them")

the orphan trace exists: 1 span
its name: 2026-08-22T06:56:23.956Z

and the real run still has: 3 spans - the tool call above is not one of them


### Mistake 4 — streaming with tools attached

**Broken on purpose.** Streaming does not run tools. Forward them anyway and the model does what
it was going to do — ask for a tool — except now the frames carry tool-call fragments instead of
prose, so the loop that prints `delta.content` prints nothing at all.

In [17]:
rendered = api("POST", f"/prompts/{PROMPT}/production/render", {"variables": {"city": "Tokyo"}})

res = http.post(
    f"{BASE_URL}/gateway/chat/completions",
    json={
        "model": rendered["model"],
        "messages": rendered["messages"],
        "tools": rendered["tools"],        # broken on purpose: tools + stream
        "stream": True,
    },
    stream=True,
    timeout=120,
)
res.raise_for_status()

text, fragments = "", 0
for line in res.iter_lines(decode_unicode=True):
    if not line or not line.startswith("data: "):
        continue
    data = line[len("data: "):]
    if data == "[DONE]":
        break
    delta = json.loads(data)["choices"][0]["delta"]
    text += delta.get("content") or ""
    if delta.get("tool_calls"):
        fragments += 1

print(f"readable text streamed: {len(text)} characters")
print(f"tool-call fragments streamed: {fragments}")
print("\nNothing to show the user, and no tool was run.")

readable text streamed: 0 characters
tool-call fragments streamed: 2

Nothing to show the user, and no tool was run.


---

## Step 12 — Close the session

**Your app.** Worth knowing what this does and does not do.

With the SDK there is a background queue: spans are reported off the critical path so they never
slow your request down, and `await hub.gateway.aclose()` flushes whatever is still waiting. There
is no queue here. Every `POST /traces` in this notebook was synchronous and already finished, so
there is nothing to flush.

What is left is the TCP connection the session is holding. Close it.

In [18]:
http.close()
print("session closed")

session closed


---

## What you built

A weather assistant over plain HTTP. The model chose when to look the weather up and which city
to ask for; your Python answered it; and all of it reads as one trace afterwards.

### What of this actually ships

The loop and its two helpers, and nothing else from this notebook:

```python
import json, os, requests
from datetime import datetime, timezone

BASE_URL = os.environ["ACRUXCORE_BASE_URL"].rstrip("/")
http = requests.Session()
http.headers.update({"Authorization": f"Bearer {os.environ['ACRUXCORE_API_KEY']}",
                     "Content-Type": "application/json"})

WEATHER = {"tokyo": "22°C, light rain", "london": "15°C, overcast", "paris": "19°C, clear"}
IMPLEMENTATIONS = {"get_weather": lambda city: {
    "city": city, "conditions": WEATHER.get(city.lower(), "no data for that city")}}


def ask_about(city, max_turns=5):
    rendered = http.post(f"{BASE_URL}/prompts/py-weather-agent/production/render",
                         json={"variables": {"city": city}}).json()
    messages, tools, trace_id = rendered["messages"], rendered["tools"], None

    for _ in range(max_turns):
        headers = {"x-trace-id": trace_id} if trace_id else {"x-trace-name": "py-weather-agent"}
        res = http.post(f"{BASE_URL}/gateway/chat/completions", headers=headers,
                        json={"model": rendered["model"], "messages": messages, "tools": tools})
        res.raise_for_status()
        trace_id = res.headers["x-gateway-trace-id"]
        message = res.json()["choices"][0]["message"]
        messages.append(message)

        if not message.get("tool_calls"):
            return message["content"]

        for call in message["tool_calls"]:
            args = json.loads(call["function"]["arguments"])
            started = datetime.now(timezone.utc).isoformat()
            result = IMPLEMENTATIONS[call["function"]["name"]](**args)
            http.post(f"{BASE_URL}/traces", json={"traces": [{
                "traceId": trace_id, "capturePayloads": True,
                "spans": [{"spanId": f"tool-{started}", "name": call["function"]["name"],
                           "kind": "tool", "status": "ok", "startTime": started,
                           "endTime": datetime.now(timezone.utc).isoformat(),
                           "input": args, "output": result}]}]})
            messages.append({"role": "tool", "tool_call_id": call["id"],
                             "content": json.dumps(result)})

    raise RuntimeError("hit the turn limit")
```

Everything else was scaffolding:

- `api` and `find_by_name` exist so this notebook can be re-run. They are notebook helpers, not
  an SDK.
- the create-and-commit cells are the dashboard's job, done once.
- every **Check** cell — the preflight, `resolve`, `render`, the trace walk — proves a step
  worked. None of it belongs in a request path.
- Step 11 is all deliberately broken.

### What this notebook left in your team

- a tool `get_weather` at v1, `production` and `staging` both pointing at it
- a prompt `py-weather-agent` at v1, with a `city` variable and `llama-3.3-70b` baked in
- a binding joining the two, inherited by every prompt alias
- several traces: the good run, the two split ones from Mistake 1, and one orphan tool span

### Where to go next

- [Build a tool-calling agent in Python (SDK)](https://docs.acruxcore.com/docs/tutorials/build-a-tool-calling-agent-in-python-sdk)
  — the same loop in one call, now that you know what it is doing for you.
- [Build a tool-calling agent in the dashboard](https://docs.acruxcore.com/docs/tutorials/build-a-tool-calling-agent-in-the-dashboard-no-code)
  — the no-code path, with HTTP tools the gateway runs for you.
- [Using sessions and traces](https://docs.acruxcore.com/docs/guides/using-sessions-and-traces)
  — group related runs and dig into what happened.